# Learning Objectives

1. ✅ Understand LlamaIndex architecture and modular ecosystem
2. ✅ Install and configure LlamaIndex
3. ✅ Configure the settings object i.e., LLM, embeddings, chunk size
4. ✅ Create first vectorstoreindex from documents
5. ✅ Execute basic queries and analyze responses
6. ✅ Understand the document -> Node -> Index flow

**Prerequisites**
* Python 3.12+ 
* Understanding of embeddings and vector similarity
* OpenAI api key

---------------------------------------------------------

# 1. Introduction to LlamaIndex

**What is LlamaIndex?**

LlamaIndex is a data framework for LLM-based applications, specifically designed to 

* **Ingest** data from various sources (PDFs, APIs, databases)
* **Index** data into optimized structures for retrieval
* **Query** data with natural language
* **Integrate** with LLMs for context-aware responses

**Why LlamaIndex Matters?**

1. **Context Window Limitations:** LLMs have token limits (~8k-128k).Llama index enables querying unlimited documents.
2. **Semantic Search:** Goes beyond key word matching using embedding-based similarity.
3. **Source Attribution:** Tracks which documents contribute to response.
4. **Production Ready:** Modular architecture, extensive integrations and active development.
   
**Architecture Overview**

Documents(Load) → Nodes(chunk) → Index(Embed) → Query Engine(Retrieve) → LLM → Response

**key Components:**

* **Documents:** Raw data sources (PDF, text, API)
* **Nodes:** Chunked text with metadata 
* **Embeddings:**  Vector representation of nodes
* **Index:** Optimized storage for retrieval (VectorStroreIndex, SummaryIndex)
* **QueryEngine:** Orchestrates retrieval and synthesis
* **Response Synthesis:** Combines retrieved context with LLM generation

---------------------------------------------------------

# 2. Installation & Modular Architecture

**New Modular Package Structure**
LlamaIndex has shifted to a three-tier architecture.

1. **llama-index-core:** Base abstractions (no integrations)
2. **Integration packages:** Specific LLMs, embeddings, vector stores
   * llama-index-llms-openai
   * llama-index-embeddings-huggingface
   * llama-index-vector-stores-qdrant
3. llama-index (meta): Bundles core + default integrations

** Why Modular?**
 * **Cherry-pick** only what you need
 * **Independent versioning** for each integration
 * **Smaller dependencies** = faster installs
 * **Future-proof** with active development

**Installation**
Dependencies are already mentioned in _requirements.txt_.

---------------------------------------------------------

# 3. Import & API Key Configuration

In [1]:
# Core llamaInex
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings, Document
from llama_index.core.node_parser import SentenceSplitter

# LLM Integration
from llama_index.llms.openai import OpenAI

# Embedding Integration
from llama_index.embeddings.openai import OpenAIEmbedding

# Utilities
from dotenv import load_dotenv
import os
import warnings
warnings.filterwarnings("ignore")

print("✅ Imports successful!")


✅ Imports successful!


# Load API keys from .env

In [2]:
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in environment variables. Please set it in the .env file.")

print(f"✅ OpenAI API key loaded (starts with: {openai_api_key[:8]}...)")

✅ OpenAI API key loaded (starts with: sk-proj-...)


---------------------------------------------------------

# 4. Global Settings Configuration

 **Understanding the Settings Object**

The settings object is modern way to configure llamaIndex globally.

**Key Configuration options:**

* **Settings.llm:** Default LLM for query engines
* **Settings.embed_model:** Default embedding model
* **Settings.chunk_size:** Default chunk size for text splitting
* **Settings.chunk_overlap:** Overlap between chunks
* **Settings.node_parser:** Default node parser

In [3]:
# configure LLM
Settings.llm = OpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
)

# configure embedding model
Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-small",
    dimensions=1536, # can be reduced for speed 
)

Settings.chunk_size = 1024
Settings.chunk_overlap = 200

Settings.node_parser = SentenceSplitter(
    chunk_size=Settings.chunk_size,
    chunk_overlap=Settings.chunk_overlap,
)

print("✅ Global Settings configured!")
print(f"    LLM: {Settings.llm.model}")
print(f"    Embedding: {Settings.embed_model.model_name}")
print(f"    Chunk Size: {Settings.chunk_size}")
print(f"    Chunk Overlap: {Settings.chunk_overlap}")

✅ Global Settings configured!
    LLM: gpt-4o-mini
    Embedding: text-embedding-3-small
    Chunk Size: 1024
    Chunk Overlap: 200


# Embedding Dimensions

text-embedding-3-small support variable dimensions.

* **1536 (default)**: Best quality, slower, more storage
* **512**: 50% faster, 67% less storage, minimal quality loss
* **256**: 75% faster, 83% less storage, noticeable quality loss

---------------------------------------------------------

# 5. Loading First Document

 **Creating Sample Data**
 We'll create sample text document about LlamaIndex. In practice, you'd load from PDF, APIs, database etc.,

In [4]:
# Create sample documents (in practice, load from files)
documents = [
    Document(
        text="""
        LlamaIndex is a data framework for large language models (LLMs). 
        It provides tools to ingest, structure, and access private or domain-specific data.
        LlamaIndex was created to solve the problem of connecting LLMs to external data sources.
        The framework supports various data sources including PDFs, databases, APIs, and web pages.
        """,
        metadata={"source": "intro", "category": "overview"}
    ),
    Document(
        text="""
        Vector embeddings are numerical representations of text that capture semantic meaning.
        In LlamaIndex, embeddings enable semantic search - finding relevant content based on meaning,
        not just keyword matching. The default embedding model is OpenAI's text-embedding-3-small,
        which produces 1536-dimensional vectors. Other models like all-MiniLM-L6-v2 produce 384 dimensions.
        """,
        metadata={"source": "embeddings", "category": "technical"}
    ),
    Document(
        text="""
        The VectorStoreIndex is the most common index type in LlamaIndex. It stores document embeddings
        in a vector database and performs similarity search during queries. When you query the index,
        it retrieves the most semantically similar chunks and passes them to the LLM as context.
        This is the foundation of Retrieval-Augmented Generation (RAG).
        """,
        metadata={"source": "vector_index", "category": "technical"}
    ),
]

print(f"✅ Created {len(documents)} sample documents")
print(f"   Total characters: {sum(len(doc.text) for doc in documents)}")

✅ Created 3 sample documents
   Total characters: 1169


# Understanding Document Objects
**Document** is the base container in LlamaIndex.

Document(
    text="...",           # The actual content
    metadata={...},       # Custom metadata (source, date, author, etc.)
    doc_id="...",        # Optional: explicit ID
)
Metadata is crucial for:

* Filtering during retrieval
* Source attribution in responses
* Provenance tracking

-------------------------------------------------------------------------------

# 6. Creating First VectorStoreIndex

** The magic: from_documents()**
This single method handles:
1. Chunking: Splits documents into nodes using `Settings.node_parser`.
2. Embedding: Generates vectors using `Settings.embed_model`.
3. Indexing: Stores in vector store(in-memory by default).

In [5]:
# Create index from documents
print("Creating VectorStoreIndex...")
print("This will:")
print("  1. Chunk documents into nodes")
print("  2. Generate embeddings for each node")
print("  3. Store in in-memory vector store\n")

index = VectorStoreIndex.from_documents(
    documents,
    show_progress=True,  # Display progress bar
)

print("\n✅ Index created successfully!")

Creating VectorStoreIndex...
This will:
  1. Chunk documents into nodes
  2. Generate embeddings for each node
  3. Store in in-memory vector store



Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/3 [00:00<?, ?it/s]


✅ Index created successfully!


# What Just Happened?
Behind the scenes:

1. **Document → Nodes**: Each document was split into smaller chunks (nodes)
2. **Nodes → Embeddings**: Each node was embedded using OpenAI's API
3. **Embeddings → Index**: Vectors were stored in SimpleVectorStore (in-memory)
**Index Types**:

`VectorStoreIndex`: Semantic similarity search (most common)
`SummaryIndex`: Sequential scanning (good for summaries)
`TreeIndex`: Hierarchical structure
`KeywordTableIndex`: Keyword extraction
`KnowledgeGraphIndex`: Entity relationships

-------------------------------------------------------------------

#  7. Basic Querying

# Creating Query Engine



In [6]:
# Query engine from index
query_engine = index.as_query_engine(
    similarity_top_k=2,  # Retrieve top 2 most similar nodes
    response_mode="compact", # compact mode returns a concise answer instead of full context
)
print("✅ Query engine created with similarity_top_k=2 and response_mode='compact'")


✅ Query engine created with similarity_top_k=2 and response_mode='compact'


# Execute First Query

In [7]:
query = "What is LlamaIndex user for?"
print(f"✅ Query: {query}")

response = query_engine.query(query)
print(f"✅ Response: {response}")

✅ Query: What is LlamaIndex user for?
✅ Response: LlamaIndex is used as a data framework for large language models (LLMs) to ingest, structure, and access private or domain-specific data. It connects LLMs to external data sources such as PDFs, databases, APIs, and web pages.


# Analyzing Response

In [8]:
print(f"Number of source nodes: {len(response.source_nodes)}")

for i, node in enumerate(response.source_nodes, 1):
    print(f"\nSource Node {i}:")
    print(f"    score: {node.score:.4f}")
    print(f"    metadata: {node.metadata}")
    print(f"    Text: {node.get_text()[:200]}...")  # Print first 200 characters of the node text
    print()

Number of source nodes: 2

Source Node 1:
    score: 0.6992
    metadata: {'source': 'intro', 'category': 'overview'}
    Text: LlamaIndex is a data framework for large language models (LLMs). 
        It provides tools to ingest, structure, and access private or domain-specific data.
        LlamaIndex was created to solve th...


Source Node 2:
    score: 0.4850
    metadata: {'source': 'vector_index', 'category': 'technical'}
    Text: The VectorStoreIndex is the most common index type in LlamaIndex. It stores document embeddings
        in a vector database and performs similarity search during queries. When you query the index,
  ...



# Similarity Scores

**Score Interpretation** - Cosine Similarity:
* **1.0**: Perfect Match
* **0.9-1.0**: Highly Relavant
* **0.7-0.9**: Relavant
* **0.5-0.7**: Somewhat Relavant
* **<0.5**: Likely unrelavant

# Experimenting different queries
# Query1: Embedding Question

In [9]:
query1 = "How do embedding work in LlamaIndex?"
print(f"✅ Query: {query1}")

response1 = query_engine.query(query1)
print(f"✅ Response: {response1}")

print("\n Top Retrieved Sources:")
print(f"    Category: {response1.source_nodes[0].metadata.get('category')}")
print(f"    Score: {response1.source_nodes[0].score:.4f}")

✅ Query: How do embedding work in LlamaIndex?
✅ Response: In LlamaIndex, embeddings function as numerical representations of text that capture their semantic meaning. This allows for semantic search, enabling the retrieval of relevant content based on meaning rather than relying solely on keyword matching. The default embedding model used is OpenAI's text-embedding-3-small, which generates 1536-dimensional vectors. Additionally, other models, such as all-MiniLM-L6-v2, can produce embeddings with 384 dimensions.

 Top Retrieved Sources:
    Category: overview
    Score: 0.6858


# Query2: RAG Specific Question

In [10]:
query2="What is Retrieval-Augmented Generation (RAG)?"
response2 = query_engine.query(query2)
print(f"✅ Query: {query2}")
print(f"✅ Response: {response2}")


✅ Query: What is Retrieval-Augmented Generation (RAG)?
✅ Response: Retrieval-Augmented Generation (RAG) is a process that involves retrieving semantically similar chunks of information from a vector database and using them as context for generating responses. This approach enhances the generation of text by incorporating relevant information based on meaning rather than relying solely on keyword matching.


# 9. Understanding the Document -> Node  -> Index Flow
# Inspecting Nodes Directly


In [11]:
# Parse the documents into nodes manually (alternative to from_documents) to understand the flow
from llama_index.core.node_parser import SentenceSplitter

parser = SentenceSplitter(chunk_size=1024, chunk_overlap=200)
nodes = parser.get_nodes_from_documents(documents)

print(f" No of nodes created from documents: {len(nodes)}\n")

for i, node in enumerate(nodes, 1):
    print(f"Node {i}:")
    print(f"    ID: {node.node_id}")
    print(f"    Text Length: {len(node.get_text())} characters")
    print(f"    Metadata: {node.metadata}")
    print(f"    Relationships: {node.relationships}")
    print()


 No of nodes created from documents: 3

Node 1:
    ID: b0301a4f-4c9c-414f-a47d-5f8c0ef8763f
    Text Length: 354 characters
    Metadata: {'source': 'intro', 'category': 'overview'}
    Relationships: {<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='ca14eb39-e868-4a20-905e-dbc1dd96a85e', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'source': 'intro', 'category': 'overview'}, hash='6b67115e521a90d22245235f7a97b2808451a01b637bc146fc4c6b1b0126d392')}

Node 2:
    ID: 17044223-1213-4c83-9dba-87e3a87f061e
    Text Length: 395 characters
    Metadata: {'source': 'embeddings', 'category': 'technical'}
    Relationships: {<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='8e6d0ff2-745a-480c-9bff-ff701dbbb719', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'source': 'embeddings', 'category': 'technical'}, hash='c74e00722500b0b510e568d1fd63750c229430501577b8d0a9295eaa48b99fb3')}

Node 3:
    ID: aa932152-5dec-4a61-8905-827d071e1bdf
    Text Length: 366 characters
    Metadat

# Node Relationships
Node track relationships:
* **Source**: Link to original document
* **Previous/Next** : Sequential Order
* **Parent/Child**: Hierarchical Structure

This enable advanced retrieval.

# Choosing Top-K
** Trade offs **

* **Low K (1-2)**: 
  * ✅ Faster queries, Lower LLM Costs (fewer tokens), 
  * ❌ May miss relavant context
* **Medium K (3-5)**:
  * ✅ Balancied retrieval, Good default for most user cases
  * ⚠️ Moderate cost/speed
* **High K (10+)**:
  * ✅ Comprehensive cost
  * ❌ Slower queries, High LLM Costs, Risk of Context Dilution

**Best Practice:** Start with k=3, tune based on your data and query complexity.

--------------------------------------------------------------------------------
# 11. Response Modes
Llamaindex supports differnt **response synthesis** strategies:

In [12]:
modes = ["compact", "tree_summarize", "simple_summarize"]
query = "What are key features of LlamaIndex?"

for mode in modes:
    print(f"\n--- Response Mode: {mode} ---")
    query_engine = index.as_query_engine(
        similarity_top_k=2,
        response_mode=mode,
    )
    response = query_engine.query(query)
    print(f"Response: {response}")
    print("--" * 40)


--- Response Mode: compact ---
Response: Key features of LlamaIndex include its ability to ingest, structure, and access private or domain-specific data, as well as its support for various data sources such as PDFs, databases, APIs, and web pages. Additionally, it utilizes the VectorStoreIndex for storing document embeddings in a vector database, enabling similarity searches and facilitating Retrieval-Augmented Generation (RAG) by providing semantically similar chunks of data to large language models during queries.
--------------------------------------------------------------------------------

--- Response Mode: tree_summarize ---
Response: Key features of LlamaIndex include its ability to ingest, structure, and access private or domain-specific data, as well as its support for various data sources such as PDFs, databases, APIs, and web pages. Additionally, it utilizes the VectorStoreIndex, which stores document embeddings in a vector database and enables similarity searches to ret

# Response Mode Comparison


| Mode | How It Works | Best For |
|---|---|---|
| compact | Concatenates chunks, refines iteratively | Balanced quality/speed |
| tree_summarize | Builds summary tree hierarchically | Large context, comprehensive answers |
| simple_summarize | Concatenates all chunks, single LLM call | Simple queries, speed |
| refine | Iteratively refines answer with each chunk | High quality, slower |
| accumulate | Generates separate answer per chunk | Multiple perspectives |

Default: `compact` - good balance for most use cases

# Next Steps in upcoming notebook:

* Loading Documents from multiple sources (PDFs, web, databases)
* Advanced chunking strategies (sentence, token, semantic)
* Metadata management and filtering
* Node relationships and hierarchies
* Optimizing chunksize for your use case
